# 00 · 高速同步开源视频模型到 5TB Google Drive

**省算力提示**：本 Notebook **无需 GPU**！请在右上角选择 **免费 CPU 运行时** 执行（消耗 `0 CU`），利用 Colab 机房千兆带宽把模型直接存进你的 5TB Google Drive (`/MyDrive/AI_Video/models/`)。

**Colab A100 40GB 默认推荐三件套（总计 ~83.5 GB，全驻留 40GB 显存零 OOM）**：
1. `wan22_i2v_fp8`（Wan 2.2 14B 图生视频 FP8 Scaled）
2. `wan22_lightx2v_lora`（LightX2V 4步加速蒸馏 LoRA）
3. `flux2_keyframe`（FLUX.2 [dev] 首帧关键帧生图主力）

In [ ]:
#@title 1. 挂载 5TB Google Drive 并拉取最新仓库代码
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

!pip install -q "huggingface_hub[hf_transfer]"
!python {REPO_DIR}/scripts/sync_models_to_drive.py --manifest {REPO_DIR}/configs/models_manifest.json --packs dummy --list


In [ ]:
#@title 2. 勾选需要同步到 5TB Google Drive 的模型包（已按 A100 40GB 预选最优组合）
WAN22_I2V_FP8 = True #@param {type:"boolean"}
WAN22_LIGHTX2V_LORA = True #@param {type:"boolean"}
FLUX2_KEYFRAME = True #@param {type:"boolean"}
WAN22_T2V_FP8 = False #@param {type:"boolean"}
LTX25_AV_A100 = False #@param {type:"boolean"}
WAN22_I2V_FP16_80GB_ONLY = False #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}
HF_TOKEN_INPUT = "" #@param {type:"string"}

import os, subprocess

# 优先读取 Colab Secrets 中的 HF_TOKEN，避免明文保存
hf_token = HF_TOKEN_INPUT.strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_token = ''

packs = []
if WAN22_I2V_FP8: packs.append('wan22_i2v_fp8')
if WAN22_LIGHTX2V_LORA: packs.append('wan22_lightx2v_lora')
if FLUX2_KEYFRAME: packs.append('flux2_keyframe')
if WAN22_T2V_FP8: packs.append('wan22_t2v_fp8')
if LTX25_AV_A100: packs.append('ltx25_av_a100')
if WAN22_I2V_FP16_80GB_ONLY: packs.append('wan22_i2v_fp16')

if not packs:
    raise ValueError('请至少勾选一个模型包！')

cmd = [
    'python', f'{REPO_DIR}/scripts/sync_models_to_drive.py',
    '--manifest', f'{REPO_DIR}/configs/models_manifest.json',
    '--packs', *packs,
]
if FORCE_REDOWNLOAD:
    cmd.append('--force')
if hf_token:
    cmd.extend(['--hf-token', hf_token])

print('执行命令:', ' '.join(cmd[:-2] + ['--hf-token', '***'] if hf_token else cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 检查 Google Drive 中已就绪的模型清单与磁盘占用
!du -sh /content/drive/MyDrive/AI_Video/models/*
!find /content/drive/MyDrive/AI_Video/models -type f -exec ls -lh {} +
